In [ ]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt
from scipy import stats
import os
import re
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore', category=UserWarning)


def safe_cox_fit(cph, df, duration_col, event_col):
    """Model fitting with adaptive regularization"""
    original_penalizer = cph.penalizer
    for penalizer in [0.1, 0.5, 1.0, 5.0]:
        cph.penalizer = penalizer
        try:
            cph.fit(df, duration_col=duration_col, event_col=event_col)
            if np.isfinite(cph.log_likelihood_):
                return True
        except:
            continue
    cph.penalizer = original_penalizer
    return False


def preprocess_data(cox_df, movement_df):
    cox_df = cox_df.copy()
    cox_df['Participant ID'] = cox_df['Participant ID'].astype(str)
    movement_df['Participant ID'] = movement_df['Participant ID'].astype(str)

    def safe_parse_time(series):
        try:
            return pd.to_datetime(
                series.str.split('~').str[0] if series.str.contains('~').any() else series,
                errors='coerce'
            ).dt.tz_localize(None)
        except:
            return pd.to_datetime(series, errors='coerce').dt.tz_localize(None)

    time_col = cox_df.columns[4]
    cox_df['start_time'] = safe_parse_time(cox_df['Start time of wear'])
    cox_df['event_time'] = safe_parse_time(cox_df[time_col])

    mask = (cox_df['start_time'].notna()) & (cox_df['event_time'].notna())
    cox_df.loc[mask, 'duration'] = (cox_df.loc[mask, 'event_time'] - cox_df.loc[mask, 'start_time']).dt.days / 365.25
    cox_df['duration'] = cox_df['duration'].fillna(10.5).clip(upper=10.5)
    cox_df['event'] = ((cox_df[time_col].notna()) &
                       (cox_df['event_time'] <= cox_df['start_time'] + pd.Timedelta(days=10.5 * 365.25))).astype(int)

    if cox_df['Sex'].dtype == object:
        cox_df['Sex'] = cox_df['Sex'].astype(str).str.upper().str.strip().map(
            {'M': 1, 'F': 0, 'MALE': 1, 'FEMALE': 0, '1': 1, '0': 0}
        ).fillna(-1)
    cox_df['Sex'] = pd.to_numeric(cox_df['Sex'], errors='coerce').fillna(-1).astype(int)

    return cox_df, movement_df


def extract_hourly_features(movement_df, disease_code=None):
    """Extract hourly features divided into 4 sets (48 hours each = 24 workday + 24 weekend)"""

    features = movement_df.iloc[:, -1:].copy()  # Only take the last column (Participant ID)
    feature_names = []

    num_total_features = movement_df.shape[1] - 1
    if num_total_features != 192:
        raise ValueError(f"Expected 192 feature columns (got {num_total_features})")

    num_sets = 4
    hours_per_set = 48  # 48 hours per set (24 workday + 24 weekend)
    hours_per_day_type = 24

    for set_idx in range(num_sets):
        set_col_indices = range(set_idx, num_total_features, 4)

        for day_type_idx, day_type in enumerate(['workday', 'weekend']):
            day_col_indices = list(set_col_indices)[
                              day_type_idx * hours_per_day_type: (day_type_idx + 1) * hours_per_day_type]

            for hour_idx, col_idx in enumerate(day_col_indices):
                feature_name = f"set{set_idx + 1}_{day_type}_hour{hour_idx:02d}"
                features[feature_name] = movement_df.iloc[:, col_idx]
                feature_names.append(feature_name)

    return features, feature_names


def analyze_hourly_cox(cox_df, features, feature_names, disease_code, output_dir):
    results = []
    analysis_df = cox_df.merge(features, on='Participant ID', how='inner')

    pattern = re.compile(r'set(\d+)_(workday|weekend)_hour(\d{2})')

    for feature in tqdm(feature_names, desc=f'Processing {disease_code}'):
        match = pattern.match(feature)
        if not match:
            continue

        set_num, day_type, hour = match.groups()
        hour = int(hour)

        if feature not in analysis_df.columns:
            continue

        analysis_subset = analysis_df[['duration', 'event', 'age', 'Sex', feature]].dropna()

        # Skip if insufficient data
        if len(analysis_subset) < 200 or analysis_subset[feature].nunique() < 5:
            continue

        try:
            # Fit Cox model
            cph = CoxPHFitter(penalizer=0.1)
            if not safe_cox_fit(cph, analysis_subset, 'duration', 'event'):
                continue

            # Get p-value for the feature
            summary = cph.summary  # 这样只获取结果而不打印
            p_value = cph.summary.p[feature]
            coef = cph.summary['coef'][feature]
            hazard_ratio = np.exp(coef)

            # Calculate median and IQR of the feature
            median_val = analysis_subset[feature].median()
            iqr_val = analysis_subset[feature].quantile(0.75) - analysis_subset[feature].quantile(0.25)

            results.append({
                'disease_code': disease_code,
                'set': set_num,
                'day_type': day_type,
                'hour': hour,
                'feature': feature,
                'n_samples': len(analysis_subset),
                'median_value': median_val,
                'iqr_value': iqr_val,
                'coef': coef,
                'hazard_ratio': hazard_ratio,
                'p_value': p_value,
                '-log10_pvalue': -np.log10(p_value) if p_value > 0 else np.nan
            })

        except Exception as e:
            print(f"Error processing {feature}: {str(e)}")
            continue

    # Save results for this disease
    if results:
        disease_results = pd.DataFrame(results)
        disease_dir = os.path.join(output_dir, disease_code)
        os.makedirs(disease_dir, exist_ok=True)
        disease_results.to_csv(os.path.join(disease_dir, f'{disease_code}_cox_results.csv'), index=False)

    return results


def main(cox_dir, movement_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    print("Loading data...")
    try:
        movement_df = pd.read_csv(movement_path)
    except Exception as e:
        print(f"Data loading failed: {str(e)}")
        return

    files = [f for f in os.listdir(cox_dir) if f.startswith('cox_5y_') and f.endswith('.csv')]
    if not files:
        print("No cox_10y_*.csv files found")
        return

    all_results = []
    for file in tqdm(files, desc='Processing diseases'):
        disease_code = file[7:-4]
        try:
            cox_df = pd.read_csv(os.path.join(cox_dir, file))
            cox_df, _ = preprocess_data(cox_df, movement_df)

            features, feature_names = extract_hourly_features(movement_df, disease_code)
            if not feature_names:
                continue

            disease_results = analyze_hourly_cox(cox_df, features, feature_names, disease_code, output_dir)
            if disease_results:
                all_results.extend(disease_results)

        except Exception as e:
            print(f"Processing {disease_code} failed: {str(e)}")
            continue

    if all_results:
        final_results = pd.DataFrame(all_results)
        final_results.to_csv(os.path.join(output_dir, 'combined_cox_results.csv'), index=False)
        print(f"Analysis completed, results saved to {os.path.join(output_dir, 'combined_cox_results.csv')}")
    else:
        print("No valid results generated")


if __name__ == '__main__':
    cox_data_dir = "../prep/all/"
    movement_data_path = "../prep/all_movement_features.csv"
    output_directory = "./cox_results_hourly"

    main(cox_data_dir, movement_data_path, output_directory)